## Requirement

In [ ]:

%pip install -r ../requirement.txt

## Pre-processing

#### Load Images for Model Training


In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_images_from_folder(dataset_path, image_size=(224, 224), target_mode=None):
    
    """
    Load images and labels from a folder structure where each subfolder is a class.
    Automatically detects image modes and converts to a consistent format.
    
    Args:
        dataset_path: Path to the dataset folder (e.g., '../assets/dataset')
        image_size: Tuple for resizing images (width, height) | Default: (224,224)
        target_mode: Target color mode ('RGB', 'L', or None for auto-detection) | Default: None
    
    Returns:
        images: Numpy array of images
        labels: Numpy array of encoded labels (it got number label from LabelEncoder. {Ex. labels = [1,2,3,2,1,3...]})
        label_encoder: Fitted LabelEncoder for decoding predictions(it knows what 0 or 1 represent. {Ex. Cat = 0, Dog = 1})
    """
    
    images = []
    labels = []
    # The set() function create a list and make sure that every items are unique
    detected_modes = set()
    
    dataset_path = Path(dataset_path)
    
    # First pass: detect modes
    for label_dir in dataset_path.iterdir():
        if label_dir.is_dir():
            for img_path in label_dir.glob('*.jpg'):
                try:
                    with Image.open(img_path) as img:
                        detected_modes.add(img.mode)
                except Exception as e:
                    print(f"Error detecting mode for {img_path}: {e}")
    
    # Determine target mode
    if target_mode is None:
        if len(detected_modes) == 1:
            target_mode = detected_modes.pop()
            print(f"All images are {target_mode} - using original format")
        else:
            target_mode = 'RGB'
            print(f"Mixed modes detected {detected_modes} - converting all to RGB")
    
    print(f"Target mode: {target_mode}")
    
    # Second pass: load images
    for label_dir in dataset_path.iterdir():
        if label_dir.is_dir():
            label = label_dir.name
            print(f"Loading images from {label}...")
            
            for img_path in label_dir.glob('*.jpg'):
                try:
                    # Load image using PIL
                    img = Image.open(img_path)
                    # Convert to target mode if needed
                    if img.mode != target_mode:
                        img = img.convert(target_mode)
                    # Resize
                    img = img.resize(image_size)
                    # Convert to numpy array
                    img_array = np.array(img)
                    images.append(img_array)
                    labels.append(label)
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")
    
    # Convert to numpy arrays for better calculation speed and utilities
    images = np.array(images)
    labels = np.array(labels)
    
    # Encode labels {LabelEncoder will look at all of the unique items in an List and assign it a number}
    label_encoder = LabelEncoder()
    labels_encoded = label_encoder.fit_transform(labels)
    
    # Print what just processed
    print(f"Loaded {len(images)} images with {len(label_encoder.classes_)} classes: {list(label_encoder.classes_)}")
    print(f"Final image shape: {images.shape}")
    
    return images, labels_encoded, label_encoder

# Using the function
images, labels, label_encoder = load_images_from_folder('../assets/dataset')

# Split into train/test (0.2 = train 80% : test 20% split)
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)
